# Final Assignment: Part 2 — Create Dashboard with Plotly and Dash
This notebook builds a sample interactive sales dashboard. Replace the sample data and charts with the exact requirements from your course if they differ.

## 1. Install dependencies

In [ ]:
%pip install pandas plotly dash

## 2. Import libraries and prepare sample data

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "Region": ["North"] * 6 + ["South"] * 6,
    "Product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Tablet"] * 2,
    "Sales": [12000, 9500, 7000, 14500, 11000, 8200,
              10500, 8700, 6500, 13000, 9800, 7600],
    "Profit": [2200, 1800, 1200, 2800, 2100, 1500,
               1900, 1600, 1000, 2400, 1850, 1350],
})

display(df.head())

## 3. Create interactive charts

In [ ]:
# Monthly sales
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales_by_month = df.groupby("Month", as_index=False)["Sales"].sum()
sales_by_month["Month"] = pd.Categorical(
    sales_by_month["Month"], categories=month_order, ordered=True
)
sales_by_month = sales_by_month.sort_values("Month")

fig_sales = px.line(
    sales_by_month, x="Month", y="Sales", markers=True,
    title="Monthly Sales", template="plotly_white"
)
fig_sales.update_layout(yaxis_title="Sales ($)", xaxis_title="Month")
fig_sales.show()

In [ ]:
# Profit by region
profit_by_region = df.groupby("Region", as_index=False)["Profit"].sum()
fig_profit = px.bar(
    profit_by_region, x="Region", y="Profit", color="Region",
    title="Profit by Region", template="plotly_white"
)
fig_profit.update_layout(yaxis_title="Profit ($)", xaxis_title="Region")
fig_profit.show()

In [ ]:
# Sales share by product
product_sales = df.groupby("Product", as_index=False)["Sales"].sum()
fig_product = px.pie(
    product_sales, names="Product", values="Sales", hole=0.35,
    title="Sales Share by Product", template="plotly_white"
)
fig_product.show()

## 4. Summary metrics

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Average Sales per Record"],
    "Value": [
        f"${df['Sales'].sum():,.0f}",
        f"${df['Profit'].sum():,.0f}",
        f"${df['Sales'].mean():,.0f}",
    ]
})
display(summary)

## 5. Optional: Run the Dash web app
Run this cell only in a local Jupyter environment. In some hosted notebooks, the app may need a tunnel or a separate terminal.

In [ ]:
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)
app.layout = html.Div([
    html.H1("Interactive Sales Dashboard"),
    html.Label("Choose region"),
    dcc.Dropdown(
        id="region-filter",
        options=[{"label": "All regions", "value": "All"}] +
                [{"label": r, "value": r} for r in sorted(df["Region"].unique())],
        value="All", clearable=False
    ),
    html.Label("Choose product"),
    dcc.Dropdown(
        id="product-filter",
        options=[{"label": "All products", "value": "All"}] +
                [{"label": p, "value": p} for p in sorted(df["Product"].unique())],
        value="All", clearable=False
    ),
    html.Div(id="metrics"),
    dcc.Graph(id="sales-chart"),
    dcc.Graph(id="profit-chart"),
    dcc.Graph(id="product-chart"),
], style={"fontFamily": "Arial", "maxWidth": "1100px", "margin": "auto", "padding": "20px"})

@app.callback(
    Output("metrics", "children"),
    Output("sales-chart", "figure"),
    Output("profit-chart", "figure"),
    Output("product-chart", "figure"),
    Input("region-filter", "value"),
    Input("product-filter", "value"),
)
def update_dashboard(region, product):
    filtered = df.copy()
    if region != "All":
        filtered = filtered[filtered["Region"] == region]
    if product != "All":
        filtered = filtered[filtered["Product"] == product]

    metrics = html.Div([
        html.H3(f"Total Sales: ${filtered['Sales'].sum():,.0f}"),
        html.H3(f"Total Profit: ${filtered['Profit'].sum():,.0f}"),
    ])

    monthly = filtered.groupby("Month", as_index=False)["Sales"].sum()
    monthly["Month"] = pd.Categorical(monthly["Month"], month_order, ordered=True)
    monthly = monthly.sort_values("Month")
    sales_fig = px.line(monthly, x="Month", y="Sales", markers=True,
                        title="Monthly Sales", template="plotly_white")

    regional = filtered.groupby("Region", as_index=False)["Profit"].sum()
    profit_fig = px.bar(regional, x="Region", y="Profit", color="Region",
                        title="Profit by Region", template="plotly_white")

    products = filtered.groupby("Product", as_index=False)["Sales"].sum()
    product_fig = px.pie(products, names="Product", values="Sales", hole=0.35,
                         title="Sales Share by Product", template="plotly_white")
    return metrics, sales_fig, profit_fig, product_fig

# Uncomment the next line to launch the Dash app:
# app.run(debug=True)